In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from ablation import run_ablation

# Polymer-segmentation ablation

One run per cell. Each `run_ablation(...)` call creates its own versioned folder
(`evaluation_truth_{model_name}_v{N}/`), trains all `n_folds`, runs TTA evaluation on the
held-out truth-test images, and writes a self-contained set of artifacts:

- `run_config.json` — the exact resolved RunConfig
- `ablation_{model}_v{N}_{timestamp}.ipynb` — copy of this notebook at execution time
- `confusion_matrix.svg` — row-normalized, editable text
- `dice_scores.md` — per-class summary + per-image table + confusion counts
- `per_image_dice.csv`
- `predictions_h5/` — predicted argmax masks (one per fold × truth-test image)
- `predictions_viz/` — palette-coloured jpgs matching `predictions_h5/`
- `checkpoints/` — `unet_fold_*_best.pth` files (scoped per run, no clashes)

The notebook is a thin orchestrator; the heavy lifting lives in the `ablation` package.
Run cells are pre-filled but commented out — uncomment whichever you want to (re)run.

In [ ]:
NOTEBOOK_PATH = Path('ablation.ipynb')
OUT_ROOT = 'evaluation_truth'

## Run A — recall-biased Tversky, vanilla U-Net (over-predicts polymer)

In [ ]:
# run_ablation(
#     model_name='unet_vanilla',
#     params=dict(
#         tversky_alpha=0.3, tversky_beta=0.7, tversky_weight=1.0,
#         fn_weight=0.3, confusion_weight=0.2,
#         directed_confusion_penalty=[(0, 1, 1.0), (0, 2, 1.0), (0, 3, 1.0)],  # symmetric (0,*)
#         selector='mean',
#     ),
#     notebook_path=NOTEBOOK_PATH,
#     out_root=OUT_ROOT,
# )

## Run B — balanced Tversky, vanilla U-Net (selected for the figure)

In [ ]:
# run_ablation(
#     model_name='unet_vanilla',
#     params=dict(
#         tversky_alpha=0.4, tversky_beta=0.6, tversky_weight=0.3,
#         fn_weight=0.1, confusion_weight=0.3,
#         directed_confusion_penalty=[(0, 1, 2.0), (0, 2, 1.0), (0, 3, 1.0)],  # asym (0,1)=2
#         selector='mean',
#     ),
#     notebook_path=NOTEBOOK_PATH,
#     out_root=OUT_ROOT,
# )

## Run C — Run B loss, ResNet-34 + ImageNet, 5-epoch encoder freeze, LR=5e-5

In [ ]:
# run_ablation(
#     model_name='smp_unet_resnet34',
#     params=dict(
#         tversky_alpha=0.4, tversky_beta=0.6, tversky_weight=0.3,
#         fn_weight=0.1, confusion_weight=0.3,
#         directed_confusion_penalty=[(0, 1, 2.0), (0, 2, 1.0), (0, 3, 1.0)],
#         lr=5e-5,
#         encoder_lr_mult=1.0,  # single LR; freeze warmup does the encoder-protection job
#         freeze_epochs=5,
#         selector='mean',
#     ),
#     notebook_path=NOTEBOOK_PATH,
#     out_root=OUT_ROOT,
# )

## Run D — Run B loss, EfficientNet-B0 + scSE, discriminative LR, floor-penalty selector

In [ ]:
# run_ablation(
#     model_name='smp_unet_effb0_scse',
#     params=dict(
#         tversky_alpha=0.4, tversky_beta=0.6, tversky_weight=0.3,
#         fn_weight=0.1, confusion_weight=0.3,
#         directed_confusion_penalty=[(0, 1, 2.0), (0, 2, 1.0), (0, 3, 1.0)],
#         lr=5e-5,
#         encoder_lr_mult=0.1,
#         freeze_epochs=0,
#         selector='floor',
#         ckpt_floor_threshold=0.70,
#         ckpt_floor_penalty_weight=0.5,
#     ),
#     notebook_path=NOTEBOOK_PATH,
#     out_root=OUT_ROOT,
# )

## Run E — Run D + recall push (β=0.7, Tversky weight=0.5)

In [ ]:
# run_ablation(
#     model_name='smp_unet_effb0_scse',
#     params=dict(
#         tversky_alpha=0.4, tversky_beta=0.7, tversky_weight=0.5,
#         fn_weight=0.1, confusion_weight=0.3,
#         directed_confusion_penalty=[(0, 1, 2.0), (0, 2, 1.0), (0, 3, 1.0)],
#         lr=5e-5,
#         encoder_lr_mult=0.1,
#         freeze_epochs=0,
#         selector='floor',
#         ckpt_floor_threshold=0.70,
#         ckpt_floor_penalty_weight=0.5,
#     ),
#     notebook_path=NOTEBOOK_PATH,
#     out_root=OUT_ROOT,
# )

## Run F — Run E config, EfficientNet-B7 + scSE (capacity control)

In [ ]:
# run_ablation(
#     model_name='smp_unet_effb7_scse',
#     params=dict(
#         tversky_alpha=0.4, tversky_beta=0.7, tversky_weight=0.5,
#         fn_weight=0.1, confusion_weight=0.3,
#         directed_confusion_penalty=[(0, 1, 2.0), (0, 2, 1.0), (0, 3, 1.0)],
#         lr=5e-5,
#         encoder_lr_mult=0.1,
#         freeze_epochs=0,
#         selector='floor',
#         ckpt_floor_threshold=0.70,
#         ckpt_floor_penalty_weight=0.5,
#     ),
#     notebook_path=NOTEBOOK_PATH,
#     out_root=OUT_ROOT,
# )